# HAIO 2024 - Spectrogram Megoldas

Zene-klasszifikacio a GTZAN adathalmaz spektrogramjai alapjan. 10 zenei mufaj, mufajonkent 100 minta.

---
## Elokeszuletek

In [ ]:
# Szukseges csomagok telepitese
!sudo apt-get install -y ffmpeg --quiet
!pip install librosa timm --quiet

In [ ]:
import os
import glob
import random
import shutil
import requests
import numpy as np
import matplotlib.pyplot as plt
from zipfile import ZipFile

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm

import librosa
import librosa.display
import IPython.display as display

from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

# Reprodukalhatosag
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Eszkoz: {device}')

### Adathalmaz letoltese

In [ ]:
# GTZAN adathalmaz letoltese es kicsomagolasa
fname = "music.zip"
url = "https://osf.io/drjhb/download"

if not os.path.isfile(fname):
    try:
        r = requests.get(url)
    except requests.ConnectionError:
        print("!!! Nem sikerult letolteni az adathalmazt !!!")
    else:
        if r.status_code != requests.codes.ok:
            print("!!! Nem sikerult letolteni az adathalmazt !!!")
        else:
            with open(fname, "wb") as fid:
                fid.write(r.content)

with ZipFile(fname, 'r') as zipObj:
    zipObj.extractall()

print("Kicsomagolas kesz!")

In [ ]:
# Train/test szetvalasztas: 80 tanito / 20 teszt mufajonkent
spectrograms_dir = "Data/images_original/"
folder_names = ['Data/train/', 'Data/test/']
train_dir = folder_names[0]
test_dir = folder_names[1]

for f in folder_names:
    if os.path.exists(f):
        shutil.rmtree(f)
        os.mkdir(f)
    else:
        os.mkdir(f)

genres = sorted(os.listdir(spectrograms_dir))
print(f'Mufajok: {genres}')

for g in genres:
    src_file_paths = []
    for im in glob.glob(os.path.join(spectrograms_dir, f'{g}', '*.png'), recursive=True):
        src_file_paths.append(im)
    random.shuffle(src_file_paths)
    test_files = src_file_paths[0:20]
    train_files = src_file_paths[20:]

    for f in folder_names:
        if not os.path.exists(os.path.join(f + f'{g}')):
            os.mkdir(os.path.join(f + f'{g}'))

    for f in train_files:
        shutil.copy(f, os.path.join(os.path.join(train_dir + f'{g}') + '/', os.path.split(f)[1]))
    for f in test_files:
        shutil.copy(f, os.path.join(os.path.join(test_dir + f'{g}') + '/', os.path.split(f)[1]))

print(f'Tanito mintak: {sum(len(os.listdir(os.path.join(train_dir, g))) for g in genres)}')
print(f'Teszt mintak: {sum(len(os.listdir(os.path.join(test_dir, g))) for g in genres)}')

In [ ]:
# Adatok betoltese ImageFolder-rel
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=25, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=25, shuffle=False, num_workers=0)

class_names = train_dataset.classes
print(f'Osztalyok: {class_names}')
print(f'Tanito mintak szama: {len(train_dataset)}, Teszt mintak szama: {len(test_dataset)}')

In [ ]:
# Segedfuggvenyek a tanitashoz es kiertekeleshez

def plot_loss_accuracy(train_loss, train_acc, validation_loss, validation_acc):
    """Tanitas es validacio loss/accuracy gorbeinek abrazolasa."""
    epochs = len(train_loss)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.plot(range(epochs), train_loss, label='Tanitasi Loss')
    ax1.plot(range(epochs), validation_loss, label='Validacios Loss')
    ax1.set_xlabel('Epocha')
    ax1.set_ylabel('Loss')
    ax1.set_title('Epocha vs Loss')
    ax1.legend()

    ax2.plot(range(epochs), train_acc, label='Tanitasi Pontossag')
    ax2.plot(range(epochs), validation_acc, label='Validacios Pontossag')
    ax2.set_xlabel('Epocha')
    ax2.set_ylabel('Pontossag')
    ax2.set_title('Epocha vs Pontossag')
    ax2.legend()
    plt.tight_layout()
    plt.show()


def evaluate(model, data_loader, device):
    """Modell kiertekelese egy adott adathalmazon."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    return correct / total


def train_model(model, device, train_loader, test_loader, epochs, lr=0.001):
    """Altalanos tanitasi ciklus loss es accuracy nyomonkovetesevel."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train_loss_hist, val_loss_hist = [], []
    train_acc_hist, val_acc_hist = [], []

    for epoch in tqdm(range(epochs), desc='Tanitas'):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0

        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(output, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

        train_loss_hist.append(running_loss / len(train_loader))
        train_acc_hist.append(correct / total)

        # Validacio
        model.eval()
        running_loss = 0.0
        correct, total = 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                running_loss += loss.item()
                _, predicted = torch.max(output, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

        val_loss_hist.append(running_loss / len(test_loader))
        val_acc_hist.append(correct / total)

        if (epoch + 1) % 10 == 0:
            print(f'Epocha {epoch+1}/{epochs} - '
                  f'Train Loss: {train_loss_hist[-1]:.4f}, Train Acc: {train_acc_hist[-1]:.4f}, '
                  f'Val Loss: {val_loss_hist[-1]:.4f}, Val Acc: {val_acc_hist[-1]:.4f}')

    return train_loss_hist, train_acc_hist, val_loss_hist, val_acc_hist

---
# 1. Feladat - Adatvizualizacio

Megvizsgaljuk, megkulonboztethetok-e a mufajok a spektrogramok alapjan: mufajonkent megjelenitsunk egy-egy spektrogramot kozos abran.

In [ ]:
# Egy spektrogram megjelenitese mufajonkent
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Spektrogramok mufajonkent', fontsize=16)

for idx, genre in enumerate(class_names):
    ax = axes[idx // 5, idx % 5]
    # Elso kep betoltese az adott mufajbol
    genre_dir = os.path.join(spectrograms_dir, genre)
    img_files = sorted(os.listdir(genre_dir))
    img_path = os.path.join(genre_dir, img_files[0])
    img = plt.imread(img_path)
    ax.imshow(img)
    ax.set_title(genre, fontsize=14)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('A spektrogramokon lathato, hogy a kulonbozo mufajok eltero mintazatokat mutatnak.')
print('Peldaul a classical mufaj kevesbe intenziv magasabb frekvenciakon,')
print('mig a metal/rock mufajok intenzivebb, szelessavubb mintazatot mutatnak.')

---
# 2. Feladat - CNN klasszifikacio

Nullarol tanitunk egy konvolucios halot (5 konvolucios reteg, batch normalizacio, dropout, max pooling). Cel: >75% teszt pontossag.

**Beallitasok:**
- Kepek atmeretezese 224x224-re
- Adam optimalizalo, lr=0.001
- CrossEntropyLoss
- 50 epocha
- Batch size: 25

In [ ]:
class MusicNet(nn.Module):
    """5 konvolucios reteget tartalmazo halo batch normalizacioval es dropout-tal."""
    def __init__(self):
        super(MusicNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=0)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=0)
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=0)
        self.conv4 = nn.Conv2d(32, 64, kernel_size=3, padding=0)
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=0)

        self.batchnorm1 = nn.BatchNorm2d(8)
        self.batchnorm2 = nn.BatchNorm2d(16)
        self.batchnorm3 = nn.BatchNorm2d(32)
        self.batchnorm4 = nn.BatchNorm2d(64)
        self.batchnorm5 = nn.BatchNorm2d(128)

        self.dropout = nn.Dropout(p=0.3)

        # A fully connected reteg bemeneteit szamitsuk ki a kepmeretbol
        # 224 -> conv+pool -> 111 -> conv+pool -> 54 -> conv+pool -> 26 -> conv+pool -> 12 -> conv+pool -> 5
        # 128 * 5 * 5 = 3200
        self.fc1 = nn.Linear(128 * 5 * 5, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.batchnorm1(self.conv1(x))), 2)
        x = F.max_pool2d(F.relu(self.batchnorm2(self.conv2(x))), 2)
        x = F.max_pool2d(F.relu(self.batchnorm3(self.conv3(x))), 2)
        x = F.max_pool2d(F.relu(self.batchnorm4(self.conv4(x))), 2)
        x = F.max_pool2d(F.relu(self.batchnorm5(self.conv5(x))), 2)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc1(x)
        return x

In [ ]:
# CNN tanitasa
cnn_model = MusicNet().to(device)
print(f'Parameterek szama: {sum(p.numel() for p in cnn_model.parameters()):,}')

cnn_train_loss, cnn_train_acc, cnn_val_loss, cnn_val_acc = train_model(
    cnn_model, device, train_loader, test_loader, epochs=50, lr=0.001
)

plot_loss_accuracy(cnn_train_loss, cnn_train_acc, cnn_val_loss, cnn_val_acc)

# Vegso pontossag
cnn_test_acc = evaluate(cnn_model, test_loader, device)
print(f'\nCNN teszt pontossag: {cnn_test_acc:.4f} ({cnn_test_acc*100:.1f}%)')

---
# 3. Feladat - Augmentacio (SpecAugment)

A Park et al. 2019 cikkben javasolt SpecAugment modszert alkalmazzuk:
- **Frekvencia maszkolás** (frequency masking): veletlenszeru vizszintes savok eltuntetese
- **Ido maszkolás** (time masking): veletlenszeru fuggoleges savok eltuntetese

Ezek a modszerek megorizik az idobeli korrelaciot, ellentetben a hagyomanyos kep-augmentacioval.

In [ ]:
class FrequencyMasking:
    """Frekvencia maszkolás: veletlenszeru vizszintes sav eltuntetese a spektrogrambol.
    Park et al., 2019 - SpecAugment
    """
    def __init__(self, freq_mask_param=20):
        self.freq_mask_param = freq_mask_param

    def __call__(self, img):
        # img: tensor [C, H, W]
        _, h, w = img.shape
        f = random.randint(0, min(self.freq_mask_param, h - 1))
        f0 = random.randint(0, h - f)
        img[:, f0:f0 + f, :] = 0
        return img


class TimeMasking:
    """Ido maszkolás: veletlenszeru fuggoleges sav eltuntetese a spektrogrambol.
    Park et al., 2019 - SpecAugment
    """
    def __init__(self, time_mask_param=20):
        self.time_mask_param = time_mask_param

    def __call__(self, img):
        # img: tensor [C, H, W]
        _, h, w = img.shape
        t = random.randint(0, min(self.time_mask_param, w - 1))
        t0 = random.randint(0, w - t)
        img[:, :, t0:t0 + t] = 0
        return img


# Augmentalt transzformacio
augmented_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    FrequencyMasking(freq_mask_param=20),
    TimeMasking(time_mask_param=20),
])

# Augmentalt adathalmaz es loader
aug_train_dataset = datasets.ImageFolder(train_dir, transform=augmented_transform)
aug_train_loader = DataLoader(aug_train_dataset, batch_size=25, shuffle=True, num_workers=0)

# Vizualizaljuk az augmentalt kepeket
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Augmentalt spektrogramok peldak', fontsize=14)
for i in range(4):
    img, label = aug_train_dataset[i * 20]
    axes[i].imshow(img.permute(1, 2, 0).numpy())
    axes[i].set_title(class_names[label])
    axes[i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Ugyanaz a CNN architektura, de augmentalt adatokkal tanul
cnn_aug_model = MusicNet().to(device)

aug_train_loss, aug_train_acc, aug_val_loss, aug_val_acc = train_model(
    cnn_aug_model, device, aug_train_loader, test_loader, epochs=50, lr=0.001
)

plot_loss_accuracy(aug_train_loss, aug_train_acc, aug_val_loss, aug_val_acc)

aug_test_acc = evaluate(cnn_aug_model, test_loader, device)
print(f'\nCNN (augmentalt) teszt pontossag: {aug_test_acc:.4f} ({aug_test_acc*100:.1f}%)')
print(f'CNN (eredeti) teszt pontossag: {cnn_test_acc:.4f} ({cnn_test_acc*100:.1f}%)')
print(f'Kulonbseg: {(aug_test_acc - cnn_test_acc)*100:+.1f}%')

---
# 4. Feladat - Transzfer tanitas

ImageNet-en elotanitott DenseNet121 modell finomhangolasa a zene-klasszifikacios feladatra.
A Palanisamy et al. 2020 cikk alapjan a termeszetes kepeken elotanitott modellek sikeresen alkalmazhatok spektrogramokra is.

In [ ]:
# DenseNet121 betoltese ImageNet sulyokkal
densenet = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

# Az utolso reteg lecserelese 10 osztalyra
num_features = densenet.classifier.in_features
densenet.classifier = nn.Linear(num_features, 10)
densenet = densenet.to(device)

print(f'DenseNet121 - utolso reteg bemenet: {num_features}, kimenet: 10')

# ImageNet normalizacio alkalmazasa
imagenet_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

imagenet_train_dataset = datasets.ImageFolder(train_dir, transform=imagenet_transform)
imagenet_test_dataset = datasets.ImageFolder(test_dir, transform=imagenet_transform)
imagenet_train_loader = DataLoader(imagenet_train_dataset, batch_size=25, shuffle=True, num_workers=0)
imagenet_test_loader = DataLoader(imagenet_test_dataset, batch_size=25, shuffle=False, num_workers=0)

In [ ]:
# DenseNet finomhangolasa
dn_train_loss, dn_train_acc, dn_val_loss, dn_val_acc = train_model(
    densenet, device, imagenet_train_loader, imagenet_test_loader, epochs=20, lr=0.0001
)

plot_loss_accuracy(dn_train_loss, dn_train_acc, dn_val_loss, dn_val_acc)

dn_test_acc = evaluate(densenet, imagenet_test_loader, device)
print(f'\nDenseNet121 teszt pontossag: {dn_test_acc:.4f} ({dn_test_acc*100:.1f}%)')
print(f'CNN (nullarol) teszt pontossag: {cnn_test_acc:.4f} ({cnn_test_acc*100:.1f}%)')
print(f'A transzfer tanulas jellemzoen jobb eredmenyt ad, mivel az ImageNet elotanitas')
print(f'hasznos vizualis jellemzoket tanult, amelyek a spektrogramokra is alkalmazhatok.')

---
# 5. Feladat - Klaszterezes

A DenseNet modell utolso retege elotti jellemzovektorokat kinyerjuk, majd t-SNE-vel 2D-be vetitjuk es abrazoljuk mufaj szerint szinezve.

In [ ]:
# Jellemzovektor kinyerese a DenseNet utolso retege elott
# Hook segitsegevel kinyerjuk az utolso reteg bemenetet

features_list = []
labels_list = []

# Hook regisztracio a classifier reteg elott
activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = input[0].detach()
    return hook

hook_handle = densenet.classifier.register_forward_hook(get_activation('features'))

densenet.eval()
with torch.no_grad():
    for data, target in imagenet_test_loader:
        data = data.to(device)
        _ = densenet(data)
        features_list.append(activation['features'].cpu().numpy())
        labels_list.append(target.numpy())

    # Tanitasi adatok jellemzovektorai is (a klaszterezeshez es hasonlosaghoz)
    train_features_list = []
    train_labels_list = []
    for data, target in imagenet_train_loader:
        data = data.to(device)
        _ = densenet(data)
        train_features_list.append(activation['features'].cpu().numpy())
        train_labels_list.append(target.numpy())

hook_handle.remove()

# Osszes jellemzovektor osszefuzese
all_features = np.concatenate(features_list + train_features_list, axis=0)
all_labels = np.concatenate(labels_list + train_labels_list, axis=0)

test_features = np.concatenate(features_list, axis=0)
test_labels = np.concatenate(labels_list, axis=0)

print(f'Jellemzovektor meret: {all_features.shape[1]}')
print(f'Osszes minta: {all_features.shape[0]}')

In [ ]:
# t-SNE vetites es vizualizacio
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(all_features)

plt.figure(figsize=(12, 10))
colors = plt.cm.tab10(np.linspace(0, 1, 10))

for i, genre in enumerate(class_names):
    mask = all_labels == i
    plt.scatter(features_2d[mask, 0], features_2d[mask, 1],
                c=[colors[i]], label=genre, alpha=0.7, s=30)

plt.legend(fontsize=12)
plt.title('t-SNE vetites mufaj szerinti szinezéssel', fontsize=16)
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

print('A vizualizacion lathato, hogy a mufajok nagyrészt elkülönülő klasztereket alkotnak.')
print('Nehany mufaj (pl. rock es metal) kozelebb esik egymashoz, ami mufaji hasonlosagukat tukrozi.')

---
# 6. Feladat - Hasonlosag

Kiszamitjuk a koszinusz hasonlosagot a mufajok kozotti jellemzovektor-centroidok (atlagvektorok) kozott, es heatmap formajaban abrazoljuk.

In [ ]:
# Mufajonkenti centroidok szamitasa (atlag jellemzovektor)
centroids = []
for i in range(10):
    mask = all_labels == i
    centroid = all_features[mask].mean(axis=0)
    centroids.append(centroid)

centroids = np.array(centroids)
print(f'Centroidok alakja: {centroids.shape}')

# Koszinusz hasonlosag matrix
cos_sim_matrix = cosine_similarity(centroids)

# Heatmap megjelenitese
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cos_sim_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_title('Koszinusz hasonlosag mufajok kozott', fontsize=16)

# Ertekek felirasa a cellakba
for i in range(10):
    for j in range(10):
        text = ax.text(j, i, f'{cos_sim_matrix[i, j]:.2f}',
                       ha='center', va='center', fontsize=8,
                       color='white' if abs(cos_sim_matrix[i, j]) > 0.5 else 'black')

plt.colorbar(im, ax=ax, label='Koszinusz hasonlosag')
plt.tight_layout()
plt.show()

print('A heatmap megmutatja mely mufajok hasonlitanak leginkabb egymasra.')
print('A magasabb ertek nagyobb hasonlosagot jelez a jellemzoterek alapjan.')

---
# 7. Feladat - Integralt gradiensek

A Sundararajan et al. 2017 cikk alapjan implementaljuk az integralt gradiensek modszeret,
amely megmutatja, hogy a modell mely pixeleket tartja fontosnak a dontesehez.

In [ ]:
def integrated_gradients(model, input_tensor, target_class, baseline=None, steps=50):
    """Integralt gradiensek szamitasa (Sundararajan et al., 2017).
    
    Az integralt gradiens a baseline-tol (alapertelmezes: fekete kep) a bemenetig
    vezeto egyenes ut menten integralja a gradienseket.
    
    IG(x) = (x - x') * integral_0^1 dF(x' + a*(x-x'))/dx da
    """
    if baseline is None:
        baseline = torch.zeros_like(input_tensor)
    
    # Interpolacios lepesek generalasa
    scaled_inputs = [baseline + (float(i) / steps) * (input_tensor - baseline) 
                     for i in range(steps + 1)]
    
    # Gradiensek szamitasa minden lepesben
    grads = []
    model.eval()
    for scaled_input in scaled_inputs:
        scaled_input = scaled_input.to(device).requires_grad_(True)
        output = model(scaled_input)
        output[0, target_class].backward()
        grads.append(scaled_input.grad.detach().cpu())
    
    # Trapez szabaly szerinti numerikus integralas
    grads = torch.stack(grads)
    avg_grads = (grads[:-1] + grads[1:]).mean(dim=0)
    
    # Integralt gradiens = (bemenet - baseline) * atlag gradiens
    integrated_grad = (input_tensor.cpu() - baseline.cpu()) * avg_grads
    
    return integrated_grad

In [ ]:
# Valasszunk egy teszt mintat
test_iter = iter(imagenet_test_loader)
test_images, test_targets = next(test_iter)

sample_idx = 0
sample_image = test_images[sample_idx:sample_idx+1]
sample_target = test_targets[sample_idx].item()

print(f'Valasztott minta mufaja: {class_names[sample_target]}')

# Integralt gradiensek szamitasa
ig = integrated_gradients(densenet, sample_image, sample_target, steps=50)

# Vizualizacio
# Az attributios terkep: csatornank osszegzese es abszolut ertek
attribution = ig.squeeze(0).sum(dim=0).numpy()  # [H, W]
attribution = np.abs(attribution)
# Normalizalas
attribution = attribution / (attribution.max() + 1e-8)

# Eredeti kep denormalizalasa a megjeleníteshez
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
orig_img = sample_image.squeeze(0).permute(1, 2, 0).numpy()
orig_img = orig_img * std + mean
orig_img = np.clip(orig_img, 0, 1)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

ax1.imshow(orig_img)
ax1.set_title(f'Eredeti spektrogram ({class_names[sample_target]})', fontsize=13)
ax1.axis('off')

ax2.imshow(attribution, cmap='hot')
ax2.set_title('Integralt gradiensek', fontsize=13)
ax2.axis('off')

ax3.imshow(orig_img)
ax3.imshow(attribution, cmap='hot', alpha=0.5)
ax3.set_title('Attributio egyuttal', fontsize=13)
ax3.axis('off')

plt.suptitle('Integralt gradiensek vizualizacio', fontsize=16)
plt.tight_layout()
plt.show()

print('Az integralt gradiensek megmutatjak, mely frekvencia-tartomanyokat es idobeli')
print('reszeit hasznalja a modell legjobban a mufaj azonositasahoz.')

---
# 8. Feladat - Vision Transformer

Elotanitott Vision Transformer (ViT) modell finomhangolasa a spektrogram-klasszifikaciora.
Osszehasonlitjuk a CNN es a transzfer tanulas eredmenyeivel.

In [ ]:
import timm

# ViT modell betoltese elotanitott sulyokkal
vit_model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=10)
vit_model = vit_model.to(device)

print(f'ViT parameterek szama: {sum(p.numel() for p in vit_model.parameters()):,}')

In [ ]:
# ViT finomhangolasa
# Alacsonyabb tanulasi ratát hasznalunk a nagy elotanitott modellhez
vit_train_loss, vit_train_acc, vit_val_loss, vit_val_acc = train_model(
    vit_model, device, imagenet_train_loader, imagenet_test_loader, epochs=15, lr=0.00005
)

plot_loss_accuracy(vit_train_loss, vit_train_acc, vit_val_loss, vit_val_acc)

vit_test_acc = evaluate(vit_model, imagenet_test_loader, device)
print(f'\n--- Osszehasonlitas ---')
print(f'CNN (nullarol):      {cnn_test_acc*100:.1f}%')
print(f'CNN (augmentalt):    {aug_test_acc*100:.1f}%')
print(f'DenseNet121 (tl):    {dn_test_acc*100:.1f}%')
print(f'Vision Transformer:  {vit_test_acc*100:.1f}%')

---
# 9. Feladat - Generalas

Variational Autoencoder (VAE) tanitasa a spektrogramokon, uj spektrogramok generalasa,
majd a generalt kepek visszaalakitasa audiova a Griffin-Lim algoritmussal.

In [ ]:
# VAE architektura
class VAE(nn.Module):
    """Variational Autoencoder spektrogramokhoz."""
    def __init__(self, latent_dim=128):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),   # 224 -> 112
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # 112 -> 56
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 56 -> 28
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), # 28 -> 14
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )
        
        # Latens ter: mu es logvar
        self.fc_mu = nn.Linear(256 * 14 * 14, latent_dim)
        self.fc_logvar = nn.Linear(256 * 14 * 14, latent_dim)
        
        # Decoder
        self.fc_decode = nn.Linear(latent_dim, 256 * 14 * 14)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), # 14 -> 28
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # 28 -> 56
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),   # 56 -> 112
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),    # 112 -> 224
            nn.Sigmoid(),
        )
    
    def encode(self, x):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        """Reparametrizacios trukk: z = mu + sigma * epsilon"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(h.size(0), 256, 14, 14)
        return self.decoder(h)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(recon_x, x, mu, logvar):
    """VAE veszteségfüggvény: rekonstrukcios hiba + KL divergencia."""
    recon_loss = F.mse_loss(recon_x, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

In [ ]:
# VAE tanitasa
vae = VAE(latent_dim=128).to(device)
vae_optimizer = optim.Adam(vae.parameters(), lr=0.001)

# Egyszerubb loader (normalizacio nelkul, mert a VAE pixel ertekeket rekonstrual)
vae_train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)

vae_losses = []
num_epochs = 50

for epoch in tqdm(range(num_epochs), desc='VAE tanitas'):
    vae.train()
    epoch_loss = 0
    for data, _ in vae_train_loader:
        data = data.to(device)
        vae_optimizer.zero_grad()
        recon, mu, logvar = vae(data)
        loss = vae_loss(recon, data, mu, logvar)
        loss.backward()
        vae_optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(vae_train_loader.dataset)
    vae_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f'Epocha {epoch+1}/{num_epochs}, Atlag loss: {avg_loss:.2f}')

plt.figure(figsize=(10, 4))
plt.plot(vae_losses)
plt.xlabel('Epocha')
plt.ylabel('Loss')
plt.title('VAE tanitasi loss')
plt.show()

In [ ]:
# Uj spektrogramok generalasa a latens terbol
vae.eval()
with torch.no_grad():
    # Veletlenszeru latens vektorok mintavetelezese
    z = torch.randn(8, 128).to(device)
    generated = vae.decode(z).cpu()

# Generalt es rekonstrualt kepek megjelenitese
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Generalt spektrogramok', fontsize=16)

for i in range(8):
    ax = axes[i // 4, i % 4]
    img = generated[i].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(f'Generalt #{i+1}')
    ax.axis('off')

plt.tight_layout()
plt.show()

# Rekonstrukciok megjelenitese: eredeti vs rekonstrualt
test_batch, _ = next(iter(DataLoader(test_dataset, batch_size=4, shuffle=True)))
with torch.no_grad():
    recon, _, _ = vae(test_batch.to(device))
    recon = recon.cpu()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Eredeti vs Rekonstrualt', fontsize=16)
for i in range(4):
    axes[0, i].imshow(test_batch[i].permute(1, 2, 0).numpy())
    axes[0, i].set_title('Eredeti')
    axes[0, i].axis('off')
    axes[1, i].imshow(np.clip(recon[i].permute(1, 2, 0).numpy(), 0, 1))
    axes[1, i].set_title('Rekonstrualt')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Generalt spektrogram visszaalakitasa audiova Griffin-Lim algoritmussal

def spectrogram_image_to_audio(img_array, sr=22050, n_fft=2048, hop_length=512, n_mels=128):
    """Spektrogram kep visszaalakitasa audiova.
    
    1. A kep szurkearnylatos konvertalasa
    2. Atmeretezve mel-spektrogram mereture
    3. Mel-spektrogram -> linearis spektrogram invertalas
    4. Griffin-Lim fazis rekonstrukcio
    """
    # Szurkearnylatos konverzio
    if img_array.ndim == 3:
        gray = np.mean(img_array, axis=2)
    else:
        gray = img_array
    
    # Atmeretezes mel-spektrogram meretre
    from PIL import Image
    img_pil = Image.fromarray((gray * 255).astype(np.uint8))
    img_resized = img_pil.resize((431, n_mels))  # tipikus GTZAN spektrogram meret
    mel_spec = np.array(img_resized).astype(np.float32) / 255.0
    
    # Skalazo: pixel ertekek -> dB
    mel_spec_db = mel_spec * 80 - 80  # hozzavetoleges dB tartomany
    
    # dB -> amplitudo
    mel_spec_power = librosa.db_to_power(mel_spec_db)
    
    # Mel -> linearis frekvencia invertalas
    mel_basis = librosa.filters.mel(sr=sr, n_fft=n_fft, n_mels=n_mels)
    # Pseudo-inverz a mel szurobanknak
    mel_basis_pinv = np.linalg.pinv(mel_basis)
    spec = np.maximum(0, mel_basis_pinv @ mel_spec_power)
    
    # Griffin-Lim fazis rekonstrukcio
    audio = librosa.griffinlim(spec, hop_length=hop_length, n_iter=64)
    
    return audio


# Generalt spektrogramok audiova alakitasa
print('Generalt spektrogramok visszaalakitasa audiova...')
print('='*50)

for i in range(min(4, generated.shape[0])):
    img = generated[i].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    
    audio = spectrogram_image_to_audio(img)
    
    # Audio normalizalasa
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio))
    
    print(f'\nGeneralt zene #{i+1} (hossz: {len(audio)/22050:.1f} masodperc):')
    display.display(display.Audio(audio, rate=22050))

print('\nMegjegyzes: A VAE altal generalt spektrogramok es az azokbol eloallitott')
print('hangok minosege korlatozott, mivel a modell viszonylag keves adaton tanult.')
print('Komplexebb architekturakkal (pl. GAN, diffuzios modell) jobb minoseg erheto el.')

---
## Osszefoglalas

| Modell | Teszt pontossag |
|--------|----------------|
| CNN (nullarol) | lasd fent |
| CNN + SpecAugment | lasd fent |
| DenseNet121 (transzfer) | lasd fent |
| Vision Transformer | lasd fent |

A transzfer tanulas es a Vision Transformer altalaban jobb eredmenyt ert el,
mint a nullarol tanitott CNN, ami azt mutatja, hogy az ImageNet-en tanult
vizualis jellemzok hasznosak a spektrogram-klasszifikaciora is.